# デモ3：写真に質問する（Qwen3-VL）

画像と質問の文を渡すと、文で答えます。画像を見て説明する、写っている文字を読む、数を数える、などが1つのモデルでできます（VLM：Vision-Language Model）。

- モデル：[Qwen/Qwen3-VL-2B-Instruct](https://huggingface.co/Qwen/Qwen3-VL-2B-Instruct)（Apache-2.0、約20億パラメータ。このシリーズで一番小さいもの）
- 中の Transformer：画像を ViT でトークンにし、そのトークンを質問の文のトークンの前に並べて、言葉の Transformer（LLM）に入れる。LLM は、答えのトークンを1つずつ予測して文を伸ばす

**最初に**：「ファイル」→「ドライブにコピーを保存」で自分のコピーを作り、「ランタイム」→「ランタイムのタイプを変更」で **GPU**（T4 など）を選んでから、上のセルから順に実行します。コードを読む必要はありません。何ができて、何が苦手かを見つけるのが目的です。

## 1. 準備

目安：約5分（うち計算 約3分。重みは約4GB）

In [ ]:
import importlib.util
import subprocess
import sys

# Colab に入っていない、または古いライブラリを入れる
if importlib.util.find_spec("transformers") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "accelerate"], check=True)

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from skimage import data

device = "cuda" if torch.cuda.is_available() else "cpu"
# T4 は bfloat16 に対応していないので、GPU では float16 を使う
dtype = torch.float16 if device == "cuda" else torch.float32
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "（GPU を選ぶと速くなります）")

In [ ]:
from transformers import AutoModelForImageTextToText, AutoProcessor

MODEL_ID = "Qwen/Qwen3-VL-2B-Instruct"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=dtype).to(device).eval()


def ask(image, question, max_new_tokens=200):
    """画像について質問し、答えの文を返す"""
    messages = [{"role": "user", "content": [{"type": "image", "image": image},
                                             {"type": "text", "text": question}]}]
    inputs = processor.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,
                                           return_dict=True, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    answer = output[:, inputs["input_ids"].shape[1]:]  # 質問の部分を除いた、答えのトークン
    return processor.batch_decode(answer, skip_special_tokens=True)[0]

## 2. 例の画像で試す

目安：約5分

In [ ]:
USE_UPLOAD = False  # True にすると、自分の画像をアップロードできる（JPEG か PNG）

# 例の画像（scikit-image に入っている写真。CC0 またはパブリックドメイン）
images = {
    "astronaut": Image.fromarray(data.astronaut()),
    "cat": Image.fromarray(data.chelsea()),
    "coffee": Image.fromarray(data.coffee()),
}
if USE_UPLOAD:
    import io
    from google.colab import files
    for name, content in files.upload().items():
        images[name] = Image.open(io.BytesIO(content)).convert("RGB")

fig, axes = plt.subplots(1, len(images), figsize=(4 * len(images), 4), squeeze=False)
for ax, (name, image) in zip(axes[0], images.items()):
    ax.imshow(image)
    ax.set_title(name)
    ax.axis("off")
plt.show()

In [ ]:
for name, image in images.items():
    print(f"--- {name} ---")
    print(ask(image, "この画像を日本語で2文で説明してください。"))

説明はもっともらしく読めますが、細かいところを確かめましょう。たとえば宇宙飛行士の写真で人の名前を答えた場合、それは正しいでしょうか（この写真は NASA の宇宙飛行士 Eileen Collins のものです）。VLM は、知らないことも、もっともらしい文で答えてしまうことがあります（ハルシネーション）。

In [ ]:
QUESTION = "宇宙飛行士の服には、何が付いていますか。"  # ← 変えてみる
print(ask(images["astronaut"], QUESTION))

## 3. やってみよう

目安：約15分

- 質問を変えます。「何匹いますか」「左にある物は何ですか」「この料理の名前は」「写っている文字を全部書き出して」など。
- 自分の写真（ホワイトボード、手書きのメモ、グラフ、実験装置など）をアップロードして、研究の場面で使えそうか試します。
- まちがった答えを探します。数え間違い、ない物を「ある」と答える（ハルシネーション）、左右の取り違えなどが起きやすいと言われています。同じ質問を英語でするとどうでしょうか。